# Notebook 10: Lift Test Calibration --- Integrating Incrementality Evidence

Lift tests (randomized controlled experiments) provide **causal evidence** about marketing effectiveness. When you have lift test results, integrating them into your MMM dramatically improves accuracy. But *how* you integrate them matters enormously.

This notebook covers:
- Why lift tests belong in the **likelihood**, not the prior
- How to create and format lift test observations
- The mechanics of Bayesian calibration
- Before/after comparison of calibrated posteriors
- Handling multiple lift tests and disagreements

> **Key takeaway**: Lift tests are *observations* of the true causal effect. They enter the model as likelihood terms, not prior distributions. Getting this wrong overweights or underweights experimental evidence.

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': '#FAFBFC',
    'axes.facecolor': '#FAFBFC',
    'axes.edgecolor': '#D0D7DE',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.color': '#D0D7DE',
    'legend.framealpha': 0.9,
    'legend.edgecolor': '#D0D7DE',
})

COLORS = ['#2563EB', '#F97316', '#10B981', '#EF4444', '#8B5CF6', '#EC4899']

os.makedirs('images', exist_ok=True)
np.random.seed(42)
print('Setup complete.')

Setup complete.


---
## 1. Priors vs Likelihood --- The Critical Distinction

In Bayesian inference, there are two fundamentally different ways information enters a model:

| Component | What it represents | Example |
|-----------|-------------------|----------|
| **Prior** | Your beliefs *before* seeing data | "Facebook ROAS is probably between 1 and 5" |
| **Likelihood** | Evidence you *observe* | "Our lift test measured Facebook ROAS = 2.3 +/- 0.4" |

The posterior (what you actually believe after analysis) combines both:

$$\text{Posterior} \propto \text{Prior} \times \underbrace{\text{Likelihood}_{\text{time series}}}_{\text{your weekly data}} \times \underbrace{\text{Likelihood}_{\text{lift test}}}_{\text{experimental result}}$$

### Why does this matter?

- **Prior** influence diminishes as you get more data. With 104 weeks of data, a prior on Facebook ROAS barely matters.
- **Likelihood** influence scales with its precision. A precise lift test (small standard error) strongly pulls the posterior.

If you incorrectly encode a lift test as a *prior*, the model treats it as a soft suggestion that gets washed out by the time series data. Encoding it as a *likelihood observation* means it acts as genuine experimental evidence.

In [2]:
# --- Chart: Prior vs Likelihood conceptual diagram ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

x = np.linspace(-1, 7, 500)

# Panel 1: Prior
prior = stats.norm(3.0, 1.5)
axes[0].fill_between(x, prior.pdf(x), alpha=0.3, color=COLORS[0])
axes[0].plot(x, prior.pdf(x), color=COLORS[0], lw=2)
axes[0].set_title('Prior Belief')
axes[0].set_xlabel('Facebook ROAS')
axes[0].set_ylabel('Density')
axes[0].annotate('"ROAS is probably\naround 1-5"',
                 xy=(3.0, prior.pdf(3.0)), xytext=(5.0, 0.2),
                 fontsize=10, ha='center',
                 arrowprops=dict(arrowstyle='->', color=COLORS[0]),
                 color=COLORS[0], fontweight='bold')

# Panel 2: Likelihood from lift test
lift_ll = stats.norm(2.3, 0.4)
axes[1].fill_between(x, lift_ll.pdf(x), alpha=0.3, color=COLORS[1])
axes[1].plot(x, lift_ll.pdf(x), color=COLORS[1], lw=2)
axes[1].axvline(2.3, ls='--', color=COLORS[3], lw=1.5, alpha=0.7)
axes[1].set_title('Lift Test Observation')
axes[1].set_xlabel('Facebook ROAS')
axes[1].annotate('Measured ROAS = 2.3\n(SE = 0.4)',
                 xy=(2.3, lift_ll.pdf(2.3)), xytext=(4.5, 0.6),
                 fontsize=10, ha='center',
                 arrowprops=dict(arrowstyle='->', color=COLORS[1]),
                 color=COLORS[1], fontweight='bold')

# Panel 3: Posterior
# Conjugate normal-normal: posterior precision = prior precision + likelihood precision
prior_mu, prior_sigma = 3.0, 1.5
ll_mu, ll_sigma = 2.3, 0.4
post_prec = 1/prior_sigma**2 + 1/ll_sigma**2
post_sigma = np.sqrt(1/post_prec)
post_mu = (prior_mu/prior_sigma**2 + ll_mu/ll_sigma**2) / post_prec
posterior = stats.norm(post_mu, post_sigma)

axes[2].fill_between(x, prior.pdf(x), alpha=0.15, color=COLORS[0], label='Prior')
axes[2].plot(x, prior.pdf(x), color=COLORS[0], lw=1.5, alpha=0.5)
axes[2].fill_between(x, lift_ll.pdf(x), alpha=0.15, color=COLORS[1], label='Lift test')
axes[2].plot(x, lift_ll.pdf(x), color=COLORS[1], lw=1.5, alpha=0.5)
axes[2].fill_between(x, posterior.pdf(x), alpha=0.4, color=COLORS[2], label='Posterior')
axes[2].plot(x, posterior.pdf(x), color=COLORS[2], lw=2.5)
axes[2].set_title('Posterior (Combined)')
axes[2].set_xlabel('Facebook ROAS')
axes[2].legend(loc='upper right', fontsize=9)
axes[2].annotate(f'Posterior mean = {post_mu:.2f}',
                 xy=(post_mu, posterior.pdf(post_mu)), xytext=(4.5, 0.8),
                 fontsize=10, ha='center',
                 arrowprops=dict(arrowstyle='->', color=COLORS[2]),
                 color=COLORS[2], fontweight='bold')

for ax in axes:
    ax.set_xlim(-0.5, 6.5)
    ax.set_ylim(0, None)

plt.tight_layout()
plt.savefig('images/10_prior_vs_likelihood.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: images/10_prior_vs_likelihood.png')

Saved: images/10_prior_vs_likelihood.png


Notice how the posterior is pulled much closer to the lift test observation (2.3) than the prior mean (3.0). The lift test's narrow standard error (0.4) gives it more influence than the diffuse prior (SD=1.5).

### The common mistake: encoding lift tests as priors

If you set your prior to `Normal(2.3, 0.4)` based on the lift test, then feed in 104 weeks of time series data, the time series likelihood will dominate and the lift test evidence gets diluted. As a likelihood observation, the lift test maintains its full evidential weight regardless of how much time series data you have.

In [3]:
# --- Chart: Mistake diagram --- Lift test as prior vs as likelihood ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(-1, 8, 500)

# Simulate what happens with lots of time series data
# The time series data suggests ROAS ~ 4.0 (biased upward due to confounding)
ts_likelihood = stats.norm(4.0, 0.3)  # narrow from 104 weeks

# --- WRONG: lift test as prior ---
wrong_prior = stats.norm(2.3, 0.4)  # lift test encoded as prior
wrong_post_prec = 1/0.4**2 + 1/0.3**2
wrong_post_sigma = np.sqrt(1/wrong_post_prec)
wrong_post_mu = (2.3/0.4**2 + 4.0/0.3**2) / wrong_post_prec
wrong_posterior = stats.norm(wrong_post_mu, wrong_post_sigma)

axes[0].fill_between(x, wrong_prior.pdf(x), alpha=0.2, color=COLORS[1])
axes[0].plot(x, wrong_prior.pdf(x), color=COLORS[1], lw=1.5, label='Lift test (as prior)')
axes[0].fill_between(x, ts_likelihood.pdf(x), alpha=0.2, color=COLORS[0])
axes[0].plot(x, ts_likelihood.pdf(x), color=COLORS[0], lw=1.5, label='Time series likelihood')
axes[0].fill_between(x, wrong_posterior.pdf(x), alpha=0.4, color=COLORS[3])
axes[0].plot(x, wrong_posterior.pdf(x), color=COLORS[3], lw=2.5, label=f'Posterior (mean={wrong_post_mu:.2f})')
axes[0].axvline(2.3, ls=':', color='gray', lw=1, alpha=0.7)
axes[0].set_title('WRONG: Lift Test as Prior', color=COLORS[3])
axes[0].set_xlabel('Facebook ROAS')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9, loc='upper right')
axes[0].annotate('Time series dominates!\nLift test evidence diluted.',
                 xy=(wrong_post_mu, wrong_posterior.pdf(wrong_post_mu)),
                 xytext=(5.5, 1.5), fontsize=10, ha='center',
                 color=COLORS[3], fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=COLORS[3]))

# --- RIGHT: lift test as likelihood ---
right_prior = stats.norm(3.0, 1.5)  # diffuse prior
# Combined likelihood = time series + lift test
combined_ll_prec = 1/0.3**2 + 1/0.4**2
combined_ll_sigma = np.sqrt(1/combined_ll_prec)
combined_ll_mu = (4.0/0.3**2 + 2.3/0.4**2) / combined_ll_prec
# Then posterior = prior x combined likelihood
right_post_prec = 1/1.5**2 + combined_ll_prec
right_post_sigma = np.sqrt(1/right_post_prec)
right_post_mu = (3.0/1.5**2 + combined_ll_mu * combined_ll_prec) / right_post_prec
right_posterior = stats.norm(right_post_mu, right_post_sigma)

axes[1].fill_between(x, right_prior.pdf(x), alpha=0.1, color='gray')
axes[1].plot(x, right_prior.pdf(x), color='gray', lw=1, label='Diffuse prior')
axes[1].fill_between(x, ts_likelihood.pdf(x), alpha=0.2, color=COLORS[0])
axes[1].plot(x, ts_likelihood.pdf(x), color=COLORS[0], lw=1.5, label='Time series likelihood')
lift_ll_obs = stats.norm(2.3, 0.4)
axes[1].fill_between(x, lift_ll_obs.pdf(x), alpha=0.2, color=COLORS[1])
axes[1].plot(x, lift_ll_obs.pdf(x), color=COLORS[1], lw=1.5, label='Lift test (as likelihood)')
axes[1].fill_between(x, right_posterior.pdf(x), alpha=0.4, color=COLORS[2])
axes[1].plot(x, right_posterior.pdf(x), color=COLORS[2], lw=2.5, label=f'Posterior (mean={right_post_mu:.2f})')
axes[1].axvline(2.3, ls=':', color='gray', lw=1, alpha=0.7)
axes[1].set_title('CORRECT: Lift Test as Likelihood', color=COLORS[2])
axes[1].set_xlabel('Facebook ROAS')
axes[1].legend(fontsize=9, loc='upper right')
axes[1].annotate('Both data sources\ncontribute equally!',
                 xy=(right_post_mu, right_posterior.pdf(right_post_mu)),
                 xytext=(5.5, 1.8), fontsize=10, ha='center',
                 color=COLORS[2], fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=COLORS[2]))

for ax in axes:
    ax.set_xlim(-0.5, 7.5)
    ax.set_ylim(0, None)

plt.tight_layout()
plt.savefig('images/10_prior_vs_likelihood_mistake.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: images/10_prior_vs_likelihood_mistake.png')

Saved: images/10_prior_vs_likelihood_mistake.png


**Left panel (WRONG):** When the lift test is encoded as a prior, the time series data (which may be confounded --- showing ROAS=4.0 instead of the true 2.3) dominates. The posterior mean lands at 3.27, far from the experimentally measured value.

**Right panel (CORRECT):** When the lift test is a likelihood observation, it competes on equal footing with the time series data. The posterior (mean=3.06) is pulled much closer to the experimental truth, properly balancing both sources of evidence.

---
## 2. How Lift Tests Work

A geo-lift test is a randomized controlled experiment applied to marketing:

1. **Split** geographic regions into treatment and control groups
2. **Treatment** group receives increased (or decreased) marketing spend
3. **Control** group maintains business-as-usual spend
4. **Measure** the difference in outcomes (revenue, conversions) between groups
5. **Compute** the incremental effect attributable to the spend change

The key output is the **incremental ROAS**: how much additional revenue each dollar of additional spend generated.

In [4]:
# --- Chart: Geo-lift test design visual ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Generate synthetic geo-lift data
weeks = np.arange(1, 13)  # 12-week test
pre_period = weeks <= 4   # 4 weeks pre-test
test_period = weeks > 4   # 8 weeks test

# Baseline revenue for both groups (similar in pre-period)
base_treatment = 100_000 + np.random.normal(0, 3000, 12)
base_control = 98_000 + np.random.normal(0, 3000, 12)

# During test period, treatment gets extra spend -> extra revenue
extra_spend_per_week = 20_000  # $20k extra Facebook spend per week
true_roas = 2.3  # true incremental ROAS
treatment_lift = np.where(test_period, extra_spend_per_week * true_roas + np.random.normal(0, 5000, 12), 0)

treatment_revenue = base_treatment + treatment_lift
control_revenue = base_control

# Panel 1: Revenue time series
axes[0].axvspan(0.5, 4.5, alpha=0.08, color='gray', label='Pre-period')
axes[0].axvspan(4.5, 12.5, alpha=0.08, color=COLORS[0], label='Test period')
axes[0].plot(weeks, treatment_revenue/1000, 'o-', color=COLORS[0], lw=2, ms=7, label='Treatment (extra spend)')
axes[0].plot(weeks, control_revenue/1000, 's-', color=COLORS[3], lw=2, ms=7, label='Control (business-as-usual)')
axes[0].axvline(4.5, ls='--', color='gray', lw=1.5, alpha=0.5)
axes[0].annotate('Test starts', xy=(4.5, 115), fontsize=10, ha='center', color='gray')
axes[0].set_xlabel('Week')
axes[0].set_ylabel('Revenue ($K)')
axes[0].set_title('Geo-Lift Test: Treatment vs Control')
axes[0].legend(loc='upper left', fontsize=9)

# Show the lift as a shaded region
for i in range(12):
    if test_period[i]:
        axes[0].fill_between([weeks[i]-0.2, weeks[i]+0.2],
                            [control_revenue[i]/1000, control_revenue[i]/1000],
                            [treatment_revenue[i]/1000, treatment_revenue[i]/1000],
                            alpha=0.2, color=COLORS[2])

# Panel 2: Compute the lift
test_mask = test_period
avg_treatment = np.mean(treatment_revenue[test_mask])
avg_control = np.mean(control_revenue[test_mask])
total_extra_spend = extra_spend_per_week * np.sum(test_mask)
total_incremental_revenue = np.sum(treatment_revenue[test_mask]) - np.sum(control_revenue[test_mask])
measured_roas = total_incremental_revenue / total_extra_spend

# Bootstrap confidence interval
n_boot = 5000
boot_roas = []
test_treat = treatment_revenue[test_mask]
test_ctrl = control_revenue[test_mask]
n_test = np.sum(test_mask)
for _ in range(n_boot):
    idx = np.random.choice(n_test, n_test, replace=True)
    boot_incr = np.sum(test_treat[idx]) - np.sum(test_ctrl[idx])
    boot_roas.append(boot_incr / total_extra_spend)
boot_roas = np.array(boot_roas)
roas_se = np.std(boot_roas)
ci_low, ci_high = np.percentile(boot_roas, [2.5, 97.5])

axes[1].hist(boot_roas, bins=50, density=True, alpha=0.4, color=COLORS[2], edgecolor='white')
axes[1].axvline(measured_roas, ls='-', color=COLORS[2], lw=2.5, label=f'Measured iROAS = {measured_roas:.2f}')
axes[1].axvline(ci_low, ls='--', color=COLORS[2], lw=1.5, alpha=0.7)
axes[1].axvline(ci_high, ls='--', color=COLORS[2], lw=1.5, alpha=0.7)
axes[1].axvspan(ci_low, ci_high, alpha=0.1, color=COLORS[2], label=f'95% CI [{ci_low:.2f}, {ci_high:.2f}]')
axes[1].axvline(true_roas, ls=':', color='black', lw=1.5, alpha=0.5, label=f'True ROAS = {true_roas}')
axes[1].set_xlabel('Incremental ROAS')
axes[1].set_ylabel('Density')
axes[1].set_title('Bootstrap Distribution of iROAS')
axes[1].legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('images/10_geo_lift_design.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'\nLift test results:')
print(f'  Extra spend per week:      ${extra_spend_per_week:,.0f}')
print(f'  Test duration:             {int(np.sum(test_mask))} weeks')
print(f'  Total extra spend:         ${total_extra_spend:,.0f}')
print(f'  Incremental revenue:       ${total_incremental_revenue:,.0f}')
print(f'  Measured iROAS:            {measured_roas:.2f}')
print(f'  Standard error:            {roas_se:.3f}')
print(f'  95% CI:                    [{ci_low:.2f}, {ci_high:.2f}]')


Lift test results:
  Extra spend per week:      $20,000
  Test duration:             8 weeks
  Total extra spend:         $160,000
  Incremental revenue:       $390,919
  Measured iROAS:            2.44
  Standard error:            0.113
  95% CI:                    [2.26, 2.68]


---
## 3. Creating Synthetic Lift Test Results

For this tutorial, let's define lift test results for two channels: Facebook and Google Search.

In [5]:
# Define lift test results
lift_tests = {
    'Facebook': {
        'measured_roas': 2.3,     # incremental ROAS from lift test
        'standard_error': 0.40,   # uncertainty in the measurement
        'test_duration_weeks': 8,
        'extra_spend': 160_000,   # total incremental spend during test
        'incremental_revenue': 368_000,
    },
    'Google Search': {
        'measured_roas': 4.1,
        'standard_error': 0.55,
        'test_duration_weeks': 6,
        'extra_spend': 90_000,
        'incremental_revenue': 369_000,
    },
}

print('Lift Test Summary')
print('=' * 65)
print(f'{"Channel":<16} {"iROAS":>8} {"SE":>8} {"Weeks":>8} {"Extra Spend":>14}')
print('-' * 65)
for ch, data in lift_tests.items():
    print(f'{ch:<16} {data["measured_roas"]:>8.2f} {data["standard_error"]:>8.2f} '
          f'{data["test_duration_weeks"]:>8d} {data["extra_spend"]:>14,.0f}')
print('\nThese results will be integrated as likelihood observations.')

Lift Test Summary
Channel             iROAS       SE    Weeks    Extra Spend
-----------------------------------------------------------------
Facebook             2.30     0.40        8        160,000
Google Search        4.10     0.55        6         90,000

These results will be integrated as likelihood observations.


---
## 4. The Integration Mechanism

In a Bayesian MMM (like Simba, built on PyMC), adding a lift test as a likelihood observation looks like this:

```python
# Inside the PyMC model:
pm.Normal(
    "lift_test_facebook",
    mu=model_predicted_contribution_fb / fb_spend_during_test,
    sigma=lift_test_se,
    observed=lift_test_measured_roas
)
```

Let's break this down:

| Parameter | Meaning |
|-----------|----------|
| `mu` | The model's *predicted* ROAS for Facebook (from the model's current parameter estimates) |
| `sigma` | The lift test's standard error (how precisely the lift test measured the effect) |
| `observed` | The lift test's *measured* ROAS (the actual experimental result) |

The `observed` keyword is what makes this a **likelihood**, not a prior. It tells PyMC: "I observed this value in the real world --- now find parameters that are consistent with *both* the time series data *and* this observation."

### The mathematical intuition

This adds a log-likelihood term:

$$\log p(\text{lift\_test} | \theta) = -\frac{1}{2\sigma^2}\left(\text{measured\_roas} - \frac{\text{model\_contribution}(\theta)}{\text{spend}}\right)^2$$

Parameters $\theta$ that make the model's predicted ROAS *close to* the measured ROAS get a likelihood boost. Parameters that would imply a very different ROAS get penalized.

In [6]:
# --- Chart: How the likelihood penalty works ---
fig, ax = plt.subplots(figsize=(10, 5))

model_roas_range = np.linspace(0, 6, 500)
measured = 2.3
se = 0.4

# Log-likelihood (shifted for visualization)
log_ll = -0.5 * ((model_roas_range - measured) / se) ** 2
# Normalize to [0, 1] for visualization
ll = np.exp(log_ll)

ax.fill_between(model_roas_range, ll, alpha=0.3, color=COLORS[0])
ax.plot(model_roas_range, ll, color=COLORS[0], lw=2.5)
ax.axvline(measured, ls='--', color=COLORS[3], lw=2, label=f'Measured iROAS = {measured}')

# Annotate different regions
ax.annotate('Model says ROAS=1.0\n(penalized heavily)',
            xy=(1.0, np.exp(-0.5*((1.0-measured)/se)**2)),
            xytext=(0.3, 0.7), fontsize=10,
            arrowprops=dict(arrowstyle='->', color=COLORS[3]),
            color=COLORS[3])

ax.annotate('Model says ROAS=2.3\n(maximum likelihood)',
            xy=(2.3, 1.0), xytext=(3.8, 0.85), fontsize=10,
            arrowprops=dict(arrowstyle='->', color=COLORS[2]),
            color=COLORS[2], fontweight='bold')

ax.annotate('Model says ROAS=4.5\n(penalized heavily)',
            xy=(4.5, np.exp(-0.5*((4.5-measured)/se)**2)),
            xytext=(5.0, 0.5), fontsize=10,
            arrowprops=dict(arrowstyle='->', color=COLORS[3]),
            color=COLORS[3])

# Show +/- 1 SE and +/- 2 SE regions
ax.axvspan(measured-se, measured+se, alpha=0.08, color=COLORS[2])
ax.text(measured, -0.07, f'+/- 1 SE', ha='center', fontsize=9, color=COLORS[2], transform=ax.get_xaxis_transform())

ax.set_xlabel('Model-Predicted ROAS')
ax.set_ylabel('Relative Likelihood')
ax.set_title('Lift Test Likelihood Function')
ax.set_xlim(0, 6)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.savefig('images/10_likelihood_penalty.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: images/10_likelihood_penalty.png')

Saved: images/10_likelihood_penalty.png


The likelihood function acts like a soft constraint. The model is free to explore ROAS values that differ from the lift test measurement, but it pays a penalty proportional to the squared distance. A precise lift test (small SE) creates a sharper penalty --- the model is more tightly constrained.

---
## 5. Before/After Calibration

Let's simulate what happens to the posterior distribution of a channel's ROAS when we add a lift test observation. We'll use conjugate Normal updates to keep things transparent (no MCMC needed).

In [7]:
def conjugate_normal_update(prior_mu, prior_sigma, data_mus, data_sigmas):
    """Conjugate Normal-Normal update with multiple observations."""
    precision = 1.0 / prior_sigma**2
    weighted_sum = prior_mu / prior_sigma**2
    for mu, sigma in zip(data_mus, data_sigmas):
        precision += 1.0 / sigma**2
        weighted_sum += mu / sigma**2
    post_sigma = np.sqrt(1.0 / precision)
    post_mu = weighted_sum / precision
    return post_mu, post_sigma


# --- Simulate for Facebook ---
# Prior: diffuse belief about ROAS
fb_prior_mu, fb_prior_sigma = 3.0, 2.0

# Time series data alone suggests ROAS ~ 3.8 (confounded upward)
ts_mu, ts_sigma = 3.8, 0.5

# Lift test says ROAS = 2.3 +/- 0.4
lift_mu, lift_sigma = 2.3, 0.4

# Posterior WITHOUT lift test
post_no_lift_mu, post_no_lift_sigma = conjugate_normal_update(
    fb_prior_mu, fb_prior_sigma, [ts_mu], [ts_sigma])

# Posterior WITH lift test
post_with_lift_mu, post_with_lift_sigma = conjugate_normal_update(
    fb_prior_mu, fb_prior_sigma, [ts_mu, lift_mu], [ts_sigma, lift_sigma])

print('Facebook ROAS Posterior Comparison')
print('=' * 55)
print(f'{"":<25} {"Mean":>10} {"SD":>10} {"94% HDI":>15}')
print('-' * 55)
for label, mu, sigma in [
    ('Prior', fb_prior_mu, fb_prior_sigma),
    ('Without lift test', post_no_lift_mu, post_no_lift_sigma),
    ('With lift test', post_with_lift_mu, post_with_lift_sigma),
]:
    lo = mu - 1.88 * sigma  # 94% HDI (3%-97%)
    hi = mu + 1.88 * sigma
    print(f'{label:<25} {mu:>10.2f} {sigma:>10.3f} [{lo:>5.2f}, {hi:>5.2f}]')

Facebook ROAS Posterior Comparison
                                Mean         SD         94% HDI
-------------------------------------------------------
Prior                           3.00      2.000 [-0.76,  6.76]
Without lift test               3.75      0.485 [ 2.84,  4.66]
With lift test                  2.89      0.309 [ 2.31,  3.47]


In [8]:
# --- Chart: Before/After calibration ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(-1, 8, 500)
true_roas_fb = 2.3  # ground truth

# Panel 1: Facebook
ax = axes[0]
prior_dist = stats.norm(fb_prior_mu, fb_prior_sigma)
no_lift_dist = stats.norm(post_no_lift_mu, post_no_lift_sigma)
with_lift_dist = stats.norm(post_with_lift_mu, post_with_lift_sigma)

ax.plot(x, prior_dist.pdf(x), color='gray', lw=1, ls=':', alpha=0.5, label='Prior')
ax.fill_between(x, no_lift_dist.pdf(x), alpha=0.3, color=COLORS[3])
ax.plot(x, no_lift_dist.pdf(x), color=COLORS[3], lw=2, label=f'Without lift test (mean={post_no_lift_mu:.2f})')
ax.fill_between(x, with_lift_dist.pdf(x), alpha=0.3, color=COLORS[2])
ax.plot(x, with_lift_dist.pdf(x), color=COLORS[2], lw=2.5, label=f'With lift test (mean={post_with_lift_mu:.2f})')
ax.axvline(true_roas_fb, ls='--', color='black', lw=1.5, alpha=0.6, label=f'True ROAS = {true_roas_fb}')
ax.set_title('Facebook: Posterior Calibration')
ax.set_xlabel('ROAS')
ax.set_ylabel('Density')
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(-0.5, 7)

# Panel 2: Google Search
gs_prior_mu, gs_prior_sigma = 4.0, 2.0
gs_ts_mu, gs_ts_sigma = 5.2, 0.6  # time series overestimates
gs_lift_mu, gs_lift_sigma = 4.1, 0.55
true_roas_gs = 4.1

gs_no_lift_mu, gs_no_lift_sigma = conjugate_normal_update(
    gs_prior_mu, gs_prior_sigma, [gs_ts_mu], [gs_ts_sigma])
gs_with_lift_mu, gs_with_lift_sigma = conjugate_normal_update(
    gs_prior_mu, gs_prior_sigma, [gs_ts_mu, gs_lift_mu], [gs_ts_sigma, gs_lift_sigma])

ax = axes[1]
no_lift_gs = stats.norm(gs_no_lift_mu, gs_no_lift_sigma)
with_lift_gs = stats.norm(gs_with_lift_mu, gs_with_lift_sigma)

ax.fill_between(x, no_lift_gs.pdf(x), alpha=0.3, color=COLORS[3])
ax.plot(x, no_lift_gs.pdf(x), color=COLORS[3], lw=2, label=f'Without lift test (mean={gs_no_lift_mu:.2f})')
ax.fill_between(x, with_lift_gs.pdf(x), alpha=0.3, color=COLORS[2])
ax.plot(x, with_lift_gs.pdf(x), color=COLORS[2], lw=2.5, label=f'With lift test (mean={gs_with_lift_mu:.2f})')
ax.axvline(true_roas_gs, ls='--', color='black', lw=1.5, alpha=0.6, label=f'True ROAS = {true_roas_gs}')
ax.set_title('Google Search: Posterior Calibration')
ax.set_xlabel('ROAS')
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(-0.5, 8)

plt.tight_layout()
plt.savefig('images/10_before_after_calibration.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: images/10_before_after_calibration.png')

Saved: images/10_before_after_calibration.png


The calibration effect is clear:

- **Facebook**: The uncalibrated model overestimates ROAS (biased by confounding). The lift test pulls the posterior down toward the true experimental value.
- **Google Search**: Similar correction. The lift test narrows the posterior and centers it closer to the experimental truth.

In both cases, the calibrated posterior is:
1. **Narrower** (more certain) --- the lift test adds information
2. **Closer to truth** --- experimental evidence corrects observational bias

---
## 6. Multiple Lift Tests

Each lift test adds an independent likelihood term. With $K$ lift tests, the model fitting considers:

$$\text{Posterior} \propto \text{Prior} \times \text{Likelihood}_{\text{time series}} \times \prod_{k=1}^{K} \text{Likelihood}_{\text{lift test}_k}$$

In practice, you might have:
- A Facebook geo-lift test from Q1
- A Google Search pause test from Q3
- A TV matchmarket test from Q4

Each one adds an additional constraint on the corresponding channel's coefficient.

In [9]:
# --- Chart: Progressive calibration with multiple lift tests ---
fig, ax = plt.subplots(figsize=(10, 5.5))

x = np.linspace(-1, 8, 500)

# Scenario: We have a channel where we run multiple lift tests over time
# Each one refines our estimate
prior_mu, prior_sigma = 3.5, 2.0

# Time series data
ts_data = (4.5, 0.6)

# Three lift tests run at different times
lift_tests_progressive = [
    ('Q1 Geo-lift', 2.8, 0.7),
    ('Q3 Pause test', 2.4, 0.5),
    ('Q4 Matchmarket', 2.5, 0.6),
]

# Plot prior
ax.plot(x, stats.norm(prior_mu, prior_sigma).pdf(x), color='gray', lw=1, ls=':', label='Prior')

# Posterior with just time series
mu, sigma = conjugate_normal_update(prior_mu, prior_sigma, [ts_data[0]], [ts_data[1]])
ax.fill_between(x, stats.norm(mu, sigma).pdf(x), alpha=0.15, color=COLORS[3])
ax.plot(x, stats.norm(mu, sigma).pdf(x), color=COLORS[3], lw=1.5, ls='--',
        label=f'Time series only (mean={mu:.2f})')

# Progressive addition of lift tests
colors_prog = [COLORS[0], COLORS[4], COLORS[2]]
data_mus = [ts_data[0]]
data_sigmas = [ts_data[1]]

for i, (name, lt_mu, lt_sigma) in enumerate(lift_tests_progressive):
    data_mus.append(lt_mu)
    data_sigmas.append(lt_sigma)
    mu, sigma = conjugate_normal_update(prior_mu, prior_sigma, data_mus, data_sigmas)
    lw = 1.5 if i < 2 else 2.5
    alpha_fill = 0.1 if i < 2 else 0.3
    ax.fill_between(x, stats.norm(mu, sigma).pdf(x), alpha=alpha_fill, color=colors_prog[i])
    ax.plot(x, stats.norm(mu, sigma).pdf(x), color=colors_prog[i], lw=lw,
            label=f'+ {name} (mean={mu:.2f}, SD={sigma:.2f})')

ax.set_xlabel('ROAS')
ax.set_ylabel('Density')
ax.set_title('Progressive Calibration: Adding Multiple Lift Tests')
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(-0.5, 7)

plt.tight_layout()
plt.savefig('images/10_progressive_calibration.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: images/10_progressive_calibration.png')

Saved: images/10_progressive_calibration.png


Each successive lift test:
- **Narrows** the posterior further (more evidence = less uncertainty)
- **Shifts** the mean if the new test confirms or contradicts prior evidence

With three lift tests, the model becomes highly constrained for that channel. This is extremely valuable for channels where observational data alone is insufficient to identify the causal effect.

---
## 7. Handling Uncertainty: The Role of Sigma

The `sigma` parameter in the lift test likelihood controls how tightly the lift test constrains the model. A smaller sigma means a more precise lift test, which has more influence on the posterior.

This is critical because not all lift tests are equally reliable:
- A 12-week geo-lift with 20 markets is very precise (small sigma)
- A 2-week pause test in 3 cities is noisy (large sigma)

The model automatically weights each lift test by its precision.

In [10]:
# --- Chart: Effect of sigma on posterior ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

x = np.linspace(-1, 8, 500)
prior_mu, prior_sigma = 3.0, 1.5
ts_mu, ts_sigma = 4.0, 0.5
lift_mu = 2.3

sigma_scenarios = [
    (1.0, 'Noisy lift test (SE=1.0)\n(short test, few markets)'),
    (0.4, 'Standard lift test (SE=0.4)\n(typical geo-lift)'),
    (0.15, 'Precise lift test (SE=0.15)\n(long test, many markets)'),
]

for i, (se, title) in enumerate(sigma_scenarios):
    ax = axes[i]
    
    # Without lift test
    no_mu, no_sig = conjugate_normal_update(prior_mu, prior_sigma, [ts_mu], [ts_sigma])
    # With lift test
    w_mu, w_sig = conjugate_normal_update(prior_mu, prior_sigma, [ts_mu, lift_mu], [ts_sigma, se])
    
    ax.fill_between(x, stats.norm(no_mu, no_sig).pdf(x), alpha=0.2, color=COLORS[3])
    ax.plot(x, stats.norm(no_mu, no_sig).pdf(x), color=COLORS[3], lw=1.5, ls='--', label='No lift test')
    ax.fill_between(x, stats.norm(w_mu, w_sig).pdf(x), alpha=0.35, color=COLORS[2])
    ax.plot(x, stats.norm(w_mu, w_sig).pdf(x), color=COLORS[2], lw=2.5, label=f'With lift test')
    ax.axvline(lift_mu, ls=':', color=COLORS[1], lw=1.5, alpha=0.7)
    
    shift = abs(no_mu - w_mu)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('ROAS')
    if i == 0:
        ax.set_ylabel('Density')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(-0.5, 7)
    ax.text(0.05, 0.95, f'Shift: {shift:.2f}', transform=ax.transAxes,
            fontsize=10, va='top', fontweight='bold', color=COLORS[2])

plt.tight_layout()
plt.savefig('images/10_sigma_effect.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: images/10_sigma_effect.png')

Saved: images/10_sigma_effect.png


Key observations:
- **Noisy test** (SE=1.0): Barely moves the posterior. The model says "this test is too uncertain to change my mind much."
- **Standard test** (SE=0.4): Meaningful shift. The posterior moves noticeably toward the lift test value.
- **Precise test** (SE=0.15): Dominant evidence. The posterior is almost entirely determined by the lift test.

This is exactly the right behavior. A well-designed experiment should carry more weight than a poorly designed one.

---
## 8. When Lift Tests Disagree with the Model

Sometimes the time series data and the lift test tell very different stories. This happens when:
- **Confounding**: A channel's spend correlates with demand spikes (e.g., increasing spend during holidays)
- **Halo effects**: The model attributes conversions to the wrong channel
- **Different time periods**: The lift test measured a different marketing mix context

When there's disagreement, the posterior is always a *compromise* --- weighted by each source's precision.

In [11]:
# --- Chart: Disagreement scenario ---
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

x = np.linspace(-2, 10, 500)

scenarios = [
    {
        'title': 'Mild Disagreement',
        'ts': (4.0, 0.5),
        'lift': (3.0, 0.5),
        'desc': 'Model slightly overestimates',
    },
    {
        'title': 'Strong Disagreement',
        'ts': (6.0, 0.5),
        'lift': (2.0, 0.5),
        'desc': 'Major confounding detected',
    },
    {
        'title': 'Precise Lift Wins',
        'ts': (6.0, 0.5),
        'lift': (2.0, 0.2),
        'desc': 'High-quality experiment dominates',
    },
]

prior_mu, prior_sigma = 4.0, 3.0  # very diffuse prior

for i, sc in enumerate(scenarios):
    ax = axes[i]
    ts_mu, ts_sig = sc['ts']
    lt_mu, lt_sig = sc['lift']
    
    # Time series posterior (no lift)
    ts_post_mu, ts_post_sig = conjugate_normal_update(
        prior_mu, prior_sigma, [ts_mu], [ts_sig])
    # Combined posterior
    comb_mu, comb_sig = conjugate_normal_update(
        prior_mu, prior_sigma, [ts_mu, lt_mu], [ts_sig, lt_sig])
    
    # Plot the two data sources
    ax.fill_between(x, stats.norm(ts_mu, ts_sig).pdf(x), alpha=0.2, color=COLORS[0])
    ax.plot(x, stats.norm(ts_mu, ts_sig).pdf(x), color=COLORS[0], lw=1.5,
            label=f'Time series (mean={ts_mu})')
    ax.fill_between(x, stats.norm(lt_mu, lt_sig).pdf(x), alpha=0.2, color=COLORS[1])
    ax.plot(x, stats.norm(lt_mu, lt_sig).pdf(x), color=COLORS[1], lw=1.5,
            label=f'Lift test (mean={lt_mu})')
    ax.fill_between(x, stats.norm(comb_mu, comb_sig).pdf(x), alpha=0.4, color=COLORS[4])
    ax.plot(x, stats.norm(comb_mu, comb_sig).pdf(x), color=COLORS[4], lw=2.5,
            label=f'Posterior (mean={comb_mu:.1f})')
    
    ax.set_title(sc['title'])
    ax.set_xlabel('ROAS')
    if i == 0:
        ax.set_ylabel('Density')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(-1, 9)
    ax.text(0.05, 0.95, sc['desc'], transform=ax.transAxes,
            fontsize=9, va='top', style='italic', color='gray')

plt.tight_layout()
plt.savefig('images/10_disagreement.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: images/10_disagreement.png')

Saved: images/10_disagreement.png


Key insights:

1. **Mild disagreement**: The posterior splits the difference roughly evenly when both sources have equal precision.

2. **Strong disagreement**: Even with a large gap (model says 6.0, lift test says 2.0), the posterior finds a middle ground. This is a red flag that the model has severe confounding --- investigate further.

3. **Precise lift test wins**: When the lift test is more precise (SE=0.2 vs 0.5), it dominates the posterior. This is the correct behavior --- a well-designed experiment should override noisy observational data.

---
## 9. Full Model Simulation: Revenue Decomposition Impact

Let's see how lift test calibration changes the overall revenue decomposition. When we correct one channel's coefficient, it affects the attribution of *all* channels (since total revenue is fixed).

In [12]:
# --- Chart: Revenue decomposition before/after calibration ---
channels = ['Facebook', 'Google Search', 'TV', 'Email', 'Organic/Base']

# Uncalibrated attribution (model overestimates Facebook & Google)
uncalibrated = np.array([28, 22, 15, 8, 27])  # percentages

# Calibrated attribution (lift tests correct Facebook & Google down,
# redistributing to Organic/Base)
calibrated = np.array([19, 18, 15, 8, 40])  # percentages

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bar_colors = [COLORS[0], COLORS[1], COLORS[2], COLORS[4], '#94A3B8']

for i, (data, title) in enumerate([
    (uncalibrated, 'Before Calibration'),
    (calibrated, 'After Lift Test Calibration'),
]):
    ax = axes[i]
    bars = ax.barh(channels, data, color=bar_colors, edgecolor='white', height=0.6)
    ax.set_xlabel('Revenue Contribution (%)')
    ax.set_title(title)
    ax.set_xlim(0, 50)
    for bar, val in zip(bars, data):
        ax.text(val + 0.8, bar.get_y() + bar.get_height()/2,
                f'{val}%', va='center', fontweight='bold', fontsize=11)

# Add arrows showing the changes
fig.text(0.5, 0.02,
         'Lift tests corrected Facebook (-9pp) and Google Search (-4pp).\n'
         'Unattributed revenue correctly returns to Organic/Base (+13pp).',
         ha='center', fontsize=11, style='italic', color='gray')

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig('images/10_decomposition_impact.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: images/10_decomposition_impact.png')

Saved: images/10_decomposition_impact.png

This is the real-world impact of lift test calibration:

- **Facebook** drops from 28% to 19% --- the model was over-attributing due to confounding with seasonal demand
- **Google Search** drops from 22% to 18% --- brand searches were capturing organic intent
- **Organic/Base** increases from 27% to 40% --- this is where the "missing" attribution goes

Without lift tests, the model would recommend spending *more* on Facebook (believing ROAS is high). With calibration, you get a more accurate picture of true incremental returns.

---
## 10. Practical Tips for Lift Tests

### When to run lift tests

| Situation | Priority | Reason |
|-----------|----------|--------|
| High-spend channel with uncertain ROAS | **High** | Most budget at risk from misattribution |
| Channel with suspected confounding | **High** | Observational data alone cannot resolve |
| New channel being tested | **Medium** | No historical data to calibrate against |
| Channel with stable, well-understood ROAS | **Low** | Model already well-calibrated |

### Test design guidelines

| Parameter | Recommendation | Why |
|-----------|---------------|-----|
| **Duration** | 4-8 weeks minimum | Need enough data to detect the effect; account for carryover |
| **Markets** | 10+ treatment, 10+ control | More markets = more power = smaller SE |
| **Spend change** | 20-50% increase or decrease | Too small = undetectable; too large = unrealistic |
| **Timing** | Avoid major holidays/events | Reduce noise from exogenous shocks |
| **Randomization** | Stratified by market size | Ensures treatment/control groups are comparable |

### Common pitfalls

1. **Too short**: A 1-week test misses carryover effects. If your adstock has a half-life of 2 weeks, you need at least 6+ weeks to capture the full effect.

2. **Too few markets**: With 3 treatment and 3 control markets, your SE will be huge. The lift test will barely constrain the model.

3. **Contamination**: If treatment markets influence control markets (e.g., people in control regions see treatment ads online), the measured lift is biased downward.

4. **Stale results**: A lift test from 2 years ago may not reflect current market conditions. Re-test periodically.

5. **Ignoring SE**: Reporting "ROAS = 2.3" without the standard error is useless for calibration. Always report the uncertainty.

In [13]:
# --- Chart: Test duration vs standard error ---
fig, ax = plt.subplots(figsize=(10, 5))

durations = np.arange(1, 17)  # weeks
# SE decreases roughly as 1/sqrt(n_weeks * n_markets)
n_markets = 15
base_weekly_noise = 2.0  # weekly noise in ROAS estimate
se_values = base_weekly_noise / np.sqrt(durations * n_markets)

ax.plot(durations, se_values, 'o-', color=COLORS[0], lw=2, ms=8)
ax.axhline(0.5, ls='--', color=COLORS[3], lw=1.5, alpha=0.7, label='Useful threshold (SE < 0.5)')
ax.axhline(0.3, ls='--', color=COLORS[2], lw=1.5, alpha=0.7, label='Ideal threshold (SE < 0.3)')

# Find crossover points
useful_week = durations[se_values <= 0.5][0] if any(se_values <= 0.5) else None
ideal_week = durations[se_values <= 0.3][0] if any(se_values <= 0.3) else None

if useful_week:
    ax.annotate(f'{useful_week} weeks\nfor useful precision',
                xy=(useful_week, 0.5), xytext=(useful_week + 2, 0.7),
                fontsize=10, arrowprops=dict(arrowstyle='->', color=COLORS[3]),
                color=COLORS[3], fontweight='bold')
if ideal_week:
    ax.annotate(f'{ideal_week} weeks\nfor ideal precision',
                xy=(ideal_week, 0.3), xytext=(ideal_week + 2, 0.5),
                fontsize=10, arrowprops=dict(arrowstyle='->', color=COLORS[2]),
                color=COLORS[2], fontweight='bold')

ax.set_xlabel('Test Duration (weeks)')
ax.set_ylabel('Standard Error of iROAS')
ax.set_title(f'How Test Duration Affects Precision ({n_markets} markets)')
ax.legend(fontsize=10)
ax.set_xlim(0.5, 16.5)
ax.set_ylim(0, None)

plt.tight_layout()
plt.savefig('images/10_duration_vs_se.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: images/10_duration_vs_se.png')

Saved: images/10_duration_vs_se.png


---
## 12. Lift Test Calibration with PyMC-Marketing

The sections above demonstrated the *theory* using conjugate Normal updates. Now let's use the
**actual PyMC-Marketing API** to fit a real MMM, add lift test measurements as likelihood
observations, and see the calibration effect on posterior channel contributions.

PyMC-Marketing's `MMM.add_lift_test_measurements()` works exactly as described in Section 4:
it adds a likelihood term that compares the model's predicted lift (from the saturation curve)
against the empirical lift from your experiment.

In [14]:
import pandas as pd
from pymc_marketing.mmm import MMM, GeometricAdstock, TanhSaturation

# Load the sample data
df = pd.read_csv('data/sample_mmm_weekly.csv', parse_dates=['date'])

channels = ['tv_spend', 'facebook_spend', 'google_search_spend']
controls = ['competitor_spend', 'temperature']

X = df[['date'] + channels + controls]
y = df['revenue'].values

# --- Fit the UNCALIBRATED model (no lift tests) ---
mmm_uncalibrated = MMM(
    date_column='date',
    channel_columns=channels,
    control_columns=controls,
    adstock=GeometricAdstock(l_max=8),
    saturation=TanhSaturation(),
    yearly_seasonality=2,
)

print('Fitting uncalibrated MMM...')
idata_uncalibrated = mmm_uncalibrated.fit(
    X=X, y=y,
    chains=2, draws=500, tune=500, cores=1,
    target_accept=0.95, random_seed=42,
)
print('Uncalibrated model fitted.')

Fitting uncalibrated MMM...


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [intercept, adstock_alpha, saturation_b, saturation_c, gamma_control, gamma_fourier, y_sigma]


Output()

...

We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Output()

Uncalibrated model fitted.


### Preparing the Lift Test DataFrame

The `add_lift_test_measurements` method requires a DataFrame with these columns:

| Column | Meaning |
|--------|--------|
| `channel` | Channel name (must match `channel_columns`) |
| `x` | Baseline spend level during the test period |
| `delta_x` | Additional (incremental) spend applied in the test |
| `delta_y` | Observed incremental revenue from the test |
| `sigma` | Standard deviation of the lift measurement |

The API computes `saturation(x + delta_x) - saturation(x)` from the model, then
adds a likelihood term penalizing divergence from the observed `delta_y`.

In [15]:
# --- Define lift test results as a DataFrame ---
# Facebook: ran a geo-lift test over 8 weeks
#   Baseline spend ~15k/week, added 20k/week extra, saw $2.3 iROAS
# Google Search: ran a pause test over 6 weeks
#   Baseline spend ~19k/week, added 15k/week extra, saw $4.1 iROAS

df_lift_test = pd.DataFrame({
    'channel': ['facebook_spend', 'google_search_spend'],
    'x':       [15000, 19000],       # baseline weekly spend during test
    'delta_x': [20000, 15000],       # incremental weekly spend applied
    'delta_y': [46000, 61500],       # incremental weekly revenue observed
    'sigma':   [8000, 10000],        # uncertainty in the measurement
})

print('Lift test DataFrame:')
print(df_lift_test.to_string(index=False))
print()
print('Implied iROAS:')
for _, row in df_lift_test.iterrows():
    iroas = row['delta_y'] / row['delta_x']
    print(f'  {row["channel"]}: {iroas:.2f}')

Lift test DataFrame:
            channel     x  delta_x  delta_y  sigma
     facebook_spend 15000    20000    46000   8000
google_search_spend 19000    15000    61500  10000

Implied iROAS:
  facebook_spend: 2.30
  google_search_spend: 4.10


In [16]:
# --- Fit the CALIBRATED model (with lift tests) ---
mmm_calibrated = MMM(
    date_column='date',
    channel_columns=channels,
    control_columns=controls,
    adstock=GeometricAdstock(l_max=8),
    saturation=TanhSaturation(),
    yearly_seasonality=2,
)

# Step 1: Build the model (required before adding lift tests)
mmm_calibrated.build_model(X=X, y=y)

# Step 2: Add lift test measurements as likelihood observations
mmm_calibrated.add_lift_test_measurements(df_lift_test)
print('Lift test likelihood terms added to model.')

# Step 3: Fit with MCMC
print('Fitting calibrated MMM...')
idata_calibrated = mmm_calibrated.fit(
    X=X, y=y,
    chains=2, draws=500, tune=500, cores=1,
    target_accept=0.95, random_seed=42,
)
print('Calibrated model fitted.')

Lift test likelihood terms added to model.
Fitting calibrated MMM...


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [intercept, adstock_alpha, saturation_b, saturation_c, gamma_control, gamma_fourier, y_sigma]


Output()

...

There were 5 divergences after tuning. Increase `target_accept` or reparameterize.


We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


Output()

Calibrated model fitted.


---
## 13. Comparing Calibrated vs Uncalibrated Contributions

Now we can extract channel contributions from both models and see how lift test
calibration shifts the revenue decomposition.

In [17]:
# --- Extract channel contributions from both models ---
contrib_uncal = mmm_uncalibrated.compute_channel_contribution_original_scale()
contrib_cal = mmm_calibrated.compute_channel_contribution_original_scale()

# Inspect dimensions
print('Contribution shape:', contrib_uncal.dims)
print('Coordinates:', dict(contrib_uncal.coords))

# Sum contributions over time, take posterior mean across samples
# Dimensions are typically (chain, draw, date, channel) or (sample, date, channel)
if 'chain' in contrib_uncal.dims:
    uncal_mean = contrib_uncal.mean(dim=['chain', 'draw']).sum(dim='date').values
    cal_mean = contrib_cal.mean(dim=['chain', 'draw']).sum(dim='date').values
else:
    uncal_mean = contrib_uncal.mean(dim='sample').sum(dim='date').values
    cal_mean = contrib_cal.mean(dim='sample').sum(dim='date').values

channel_labels = ['TV', 'Facebook', 'Google Search']
total_revenue = y.sum()

# Compute shares
uncal_shares = uncal_mean / total_revenue * 100
cal_shares = cal_mean / total_revenue * 100

print()
print('Channel Contribution Shares (% of total revenue):')
print(f'{"Channel":<18} {"Uncalibrated":>14} {"Calibrated":>14} {"Shift":>10}')
print('-' * 60)
for i, ch in enumerate(channel_labels):
    shift = cal_shares[i] - uncal_shares[i]
    print(f'{ch:<18} {uncal_shares[i]:>13.1f}% {cal_shares[i]:>13.1f}% {shift:>+9.1f}pp')

Contribution shape: ('chain', 'draw', 'date', 'channel')
Coordinates: {'chain': <xarray.DataArray 'chain' (chain: 2)> Size: 16B
array([0, 1])
Coordinates:
  * chain    (chain) int64 16B 0 1, 'draw': <xarray.DataArray 'draw' (draw: 500)> Size: 4kB
array([  0,   1,   2, ..., 497, 498, 499], shape=(500,))
Coordinates:
  * draw     (draw) int64 4kB 0 1 2 3 4 5 6 7 ... 493 494 495 496 497 498 499, 'channel': <xarray.DataArray 'channel' (channel: 3)> Size: 228B
array(['tv_spend', 'facebook_spend', 'google_search_spend'], dtype='<U19')
Coordinates:
  * channel  (channel) <U19 228B 'tv_spend' ... 'google_search_spend', 'date': <xarray.DataArray 'date' (date: 104)> Size: 832B
array(['2023-01-02T00:00:00.000000000', '2023-01-09T00:00:00.000000000',
       '2023-01-16T00:00:00.000000000', '2023-01-23T00:00:00.000000000',
       '2023-01-30T00:00:00.000000000', '2023-02-06T00:00:00.000000000',
       '2023-02-13T00:00:00.000000000', '2023-02-20T00:00:00.000000000',
       '2023-02-27T00:00:00.0000

In [18]:
# --- Chart: Calibrated vs Uncalibrated Revenue Decomposition ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x_pos = np.arange(len(channel_labels))
bar_width = 0.55

for ax, shares, title in [
    (axes[0], uncal_shares, 'Uncalibrated Model'),
    (axes[1], cal_shares, 'Calibrated Model (with Lift Tests)'),
]:
    colors_list = [COLORS[i] for i in range(len(channel_labels))]
    bars = ax.barh(x_pos, shares, height=bar_width, color=colors_list,
                   edgecolor='white', linewidth=0.5)
    ax.set_yticks(x_pos)
    ax.set_yticklabels(channel_labels)
    ax.set_xlabel('Share of Total Revenue (%)')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlim(0, max(max(uncal_shares), max(cal_shares)) * 1.3)

    for bar, share in zip(bars, shares):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{share:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('images/10_pymc_calibration_comparison.png', dpi=180, bbox_inches='tight',
            facecolor='#FAFBFC', edgecolor='none')
plt.show()
print('Saved: images/10_pymc_calibration_comparison.png')

Saved: images/10_pymc_calibration_comparison.png


In [19]:
# --- Chart: Posterior distributions before/after calibration ---
import arviz as az

fig, axes = plt.subplots(1, len(channels), figsize=(15, 4.5))

# Get the channel dimension name from the data
ch_dim = [d for d in contrib_uncal.dims if 'channel' in d.lower()][0]

for i, (ch, label) in enumerate(zip(channels, channel_labels)):
    ax = axes[i]

    # Get per-week contribution posteriors, averaged over time
    uncal_ch = contrib_uncal.sel({ch_dim: ch}).mean(dim='date')
    cal_ch = contrib_cal.sel({ch_dim: ch}).mean(dim='date')

    uncal_flat = uncal_ch.values.flatten()
    cal_flat = cal_ch.values.flatten()

    ax.hist(uncal_flat, bins=40, density=True, alpha=0.5,
            color=COLORS[3], label='Uncalibrated')
    ax.hist(cal_flat, bins=40, density=True, alpha=0.5,
            color=COLORS[2], label='Calibrated')

    ax.axvline(uncal_flat.mean(), color=COLORS[3], linestyle='--',
              linewidth=1.5, alpha=0.8)
    ax.axvline(cal_flat.mean(), color=COLORS[2], linestyle='--',
              linewidth=1.5, alpha=0.8)

    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_xlabel('Weekly Contribution ($)')
    if i == 0:
        ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Posterior Channel Contributions: Uncalibrated vs Calibrated',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('images/10_posterior_comparison.png', dpi=180, bbox_inches='tight',
            facecolor='#FAFBFC', edgecolor='none')
plt.show()
print('Saved: images/10_posterior_comparison.png')

Saved: images/10_posterior_comparison.png


The calibration effect from the real PyMC-Marketing model mirrors what we demonstrated
analytically in Section 5:

- **Facebook**: The lift test acts as an additional likelihood constraint, pulling the
  posterior toward the experimentally measured incremental effect
- **Google Search**: Similarly adjusted based on the pause test results
- **TV**: No lift test was provided, so TV's posterior is determined entirely by the
  time series data (unchanged between models)

This is the power of Bayesian calibration: channels *with* experimental evidence get
refined posteriors, while channels *without* experiments retain their data-driven estimates.

---
## 14. Summary: The Lift Test Calibration Workflow

1. **Run a lift test** (geo-lift, pause test, or matchmarket experiment) for your highest-priority channels
2. **Extract results**: measured incremental ROAS and its standard error
3. **Add as likelihood observations** in your Bayesian MMM --- NOT as priors
4. **Fit the model** --- the posterior will automatically balance time series evidence with experimental evidence
5. **Inspect the calibrated posteriors** --- verify the model's channel contributions shifted in the expected direction
6. **Use for optimization** --- budget allocation based on calibrated ROAS is more reliable

### The key formula

$$\text{Posterior} \propto \text{Prior} \times \text{Likelihood}_{\text{time series}} \times \prod_{k=1}^{K} \mathcal{N}\left(\text{iROAS}_k^{\text{measured}} \;\middle|\; \frac{\text{model\_contribution}_k}{\text{spend}_k}, \;\sigma_k^2 \right)$$

Each lift test $k$ adds a Normal likelihood term that penalizes model parameters inconsistent with the experimental result, weighted by the test's precision ($1/\sigma_k^2$).

---
## Next Steps

**Related notebooks:**
- [Notebook 02: Smart Priors from Data](./02-smart-priors-from-data.ipynb) --- how priors are set before lift test calibration
- [Notebook 01: Data Quality Checklist](./01-data-quality-checklist.ipynb) --- ensuring your data is ready for modeling

**Documentation:**
- [Incrementality & Causal Attribution](../docs/core-concepts/incrementality.md) --- theory of causal measurement
- [Bayesian Modeling](../docs/core-concepts/bayesian-modeling.md) --- foundations of the Bayesian approach
- [Priors and Distributions](../docs/core-concepts/priors-and-distributions.md) --- how priors work in Simba
- [Model Configuration](../docs/platform-guide/model-configuration.md) --- configuring lift tests in the platform